# Ilera-ATLAS · 02_draft: N-ATLaS drafts Q&A from the Standing Orders

**Runbook Part 3 · Sat 3 Oct.** N-ATLaS reads each CHEW chunk from `01_extract` and drafts
questions with structured answers (`ACTION / WHAT TO DO / DANGER SIGNS / SOURCE`). Humans fix them in
Google Sheets afterwards (Part 4).

Run it **twice**:

| Run | `PASS` | When | Output |
|---|---|---|---|
| 1 | `"vignette"` | Sat morning | case stories: ~650 benchmark candidates + ~60 probe items + training vignettes |
| 2 | `"qa"` | Sat night, **Commit mode** | direct questions for training only (probe sections skipped) |

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · secret `HF_TOKEN` attached.

**First do a smoke test:** set `LIMIT = 4`, run all, read the printed drafts. If they look right,
set `LIMIT = None` and use **Save Version → Save & Run All (Commit)** for the full run.
The notebook saves progress to Hugging Face every 20 batches and **resumes** where it stopped if re-run.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"          # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"
PASS = "vignette"                      # "vignette" (run 1) or "qa" (run 2)
LIMIT = 4                              # smoke test: first 4 chunks. Set None for the full run.
N_VIGNETTE, N_QA = 10, 12              # items requested per chunk
PROBE_TARGET = 90                      # probe drafts wanted in total (about 60 survive review)
BATCH = 4                              # lower to 2 if you get CUDA out of memory
MAX_NEW = 1800                         # max generated tokens per chunk
SAVE_EVERY = 20                        # batches between progress uploads
UPLOAD = True                          # False = keep everything local
DATA_DIR = None                        # local folder with chunks.jsonl + probe_chunks.txt (offline testing only)

In [ ]:
import os
import subprocess, sys
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # deprecated Kaggle default -> removes the FutureWarning
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"         # its replacement: faster model downloads
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate"], capture_output=True, text=True)
if r.returncode != 0:                               # pip prints harmless dependency warnings; show them only on failure
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install failed")
print("packages installed")
# If the next cell fails with an import error: Run > Restart kernel, then run all cells
# again (this install cell is then a quick no-op).

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    import glob                                     # API runs: token in a private Kaggle dataset
    for p in glob.glob("/kaggle/input/**/hf_token.txt", recursive=True):
        return open(p).read().strip()
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or DATA_DIR is None:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Load chunks and probe list
Only chunks with `draft = true` (CHEW) are used. The probe file must exist: probe sections get extra
vignettes in run 1 and **no** Q&A at all in run 2.

In [ ]:
import json, random, re, time

def data_file(name):
    if DATA_DIR:
        return os.path.join(DATA_DIR, name)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(DATA_REPO, f"raw/{name}", repo_type="dataset", token=HF_TOKEN)

chunks = [json.loads(l) for l in open(data_file("chunks.jsonl"), encoding="utf-8")]
chunks = [c for c in chunks if c["draft"]]
probe = set(open(data_file("probe_chunks.txt"), encoding="utf-8").read().split())
assert probe, ("probe_chunks.txt is empty: pick PROBE_SECTIONS in 01_extract and re-run its last "
               "cells before drafting, otherwise probe topics leak into training")
print(len(chunks), "drafting chunks |", len([c for c in chunks if c["chunk_id"] in probe]), "of them are probe chunks")

## 2. System prompt and drafting prompts
`SYSTEM` is the single answer-format contract used everywhere (training, evaluation, API). It is
saved to `config/system_prompt.txt` on Hugging Face so later notebooks load the exact same text.

In [ ]:
SYSTEM = ("You are Ilera-ATLAS, a primary health care assistant for trained Nigerian "
          "community health workers. Follow the 2024 National Standing Orders. "
          "Answer in the same language as the question. Use exactly this format:\n"
          "ACTION: TREAT | REFER | URGENT\n"
          "WHAT TO DO: numbered steps\n"
          "DANGER SIGNS: signs that mean urgent referral\n"
          "SOURCE: document and section")

LAYOUT = """Use this exact layout for each one, and end each one with ### on its own line:

Q: <{q_hint}>
ACTION: <one word: TREAT if the health worker manages it at the PHC, REFER if the extract says to refer, URGENT if it is an emergency needing immediate referral>
WHAT TO DO:
1. <step>
2. <step>
DANGER SIGNS: <signs that mean urgent referral, or "none listed">
###

Example:
Q: {example_q}
ACTION: REFER
WHAT TO DO:
1. Give the first dose of the pre-referral treatment listed in the extract.
2. Write a referral note and send the patient to the nearest health facility today.
DANGER SIGNS: convulsions, unable to drink, very sleepy
###"""

QA_PROMPT = """Read this extract from the Nigerian National Standing Orders ({source}).

{text}

Write {n} different questions a community health worker might ask about this extract, each with the
correct answer based ONLY on the extract. Do not invent drugs, doses or steps that are not in the extract.
""" + LAYOUT.replace("{q_hint}", "question").replace(
    "{example_q}", "What should I do for a child with fever and fast breathing?")

VIGNETTE_PROMPT = """Read this extract from the Nigerian National Standing Orders ({source}).

{text}

Write {n} different short patient case stories (age, sex, main symptoms, for how long) that a community
health worker might meet, where this extract tells them what to do. Each case ends with the question
"What should I do?". Mix mild, moderate and severe cases. Answer based ONLY on the extract. Do not invent
drugs, doses or steps that are not in the extract.
""" + LAYOUT.replace("{q_hint}", "case story ending with: What should I do?").replace(
    "{example_q}", "A 2-year-old boy has had fever for 3 days and is breathing fast. What should I do?")

print(VIGNETTE_PROMPT.format(source="<source>", text="<chunk text>", n=8)[:1200])

## 3. Parser
Turns the model's text into rows. It is forgiving about small format slips (`Action: Refer`,
`URGENT REFERRAL`, missing final `###`) and drops anything without a question, an ACTION and steps.
The SOURCE line is added from the chunk, never trusted from the model.

In [ ]:
BLOCK_SPLIT = re.compile(r"\n\s*###\s*\n?|\n(?=Q\s*[:.])")

def norm_action(s):
    s = s.upper()
    if "URGENT" in s or "EMERGENCY" in s or "IMMEDIATE" in s:
        return "URGENT"
    if "REFER" in s:
        return "REFER"
    if "TREAT" in s or "MANAGE" in s:
        return "TREAT"
    return None

def normalize(raw):
    # map Markdown-style answers (### Case 1, **Case Story:**, **Action:** ...) to the plain layout
    t = raw.replace("**", "").replace("__", "")
    t = re.sub(r"^\s*#{1,4}\s*(case|question|scenario)?\s*\d*\s*:?\s*$", "###", t, flags=re.M | re.I)
    t = re.sub(r"^\s*(case story|patient case|case|scenario|question)\s*\d*\s*[:.]\s*", "Q: ", t, flags=re.M | re.I)
    t = re.sub(r"^\s*#{0,4}\s*action\s*:", "ACTION:", t, flags=re.M | re.I)
    t = re.sub(r"^\s*#{0,4}\s*what to do\s*:", "WHAT TO DO:", t, flags=re.M | re.I)
    t = re.sub(r"^\s*#{0,4}\s*danger signs?\s*:", "DANGER SIGNS:", t, flags=re.M | re.I)
    return t

ACTION_LINE = re.compile(r"^\s*ACTION\s*:\s*(.+?)\s*$", re.M | re.I)
STEP_LINE = re.compile(r"^\s*(\d+[.)]|[-•*])\s+\S")

def parse(raw, chunk, kind):
    # Anchored on each ACTION line: the question is the paragraph just before it, the answer is the
    # numbered steps + DANGER SIGNS just after it. Works for "Q:" blocks, Markdown "### Case n" blocks
    # and case stories without any prefix or separator.
    t = normalize(raw)
    acts = list(ACTION_LINE.finditer(t))
    items, seen, prev_end = [], set(), 0
    for i, a in enumerate(acts):
        nxt = acts[i + 1].start() if i + 1 < len(acts) else len(t)
        before = t[prev_end:a.start()]
        paras = [p for p in re.split(r"\n\s*\n|^\s*#{1,4}\s*$", before, flags=re.M) if p and p.strip()]
        question = " ".join(re.sub(r"^\s*(#{1,4}\s*)?(Q\s*[:.]\s*)?", "", paras[-1] if paras else "").split())
        after = t[a.end():nxt]
        wtd = re.search(r"^\s*WHAT TO DO\s*:?(.*)$", after, re.M | re.I)
        step_lines, end = [], 0
        if wtd:
            if wtd.group(1).strip():
                step_lines.append(wtd.group(1).strip())
            end = wtd.end()
            for line in after[wtd.end():].splitlines(keepends=True):
                if STEP_LINE.match(line):
                    step_lines.append(line.strip()); end += len(line)
                elif not line.strip() and not step_lines:
                    end += len(line)
                elif line.strip() and step_lines and not re.match(r"^\s*(DANGER|###|Q\s*[:.])", line, re.I) \
                        and len(line.strip()) < 200 and not line.strip().endswith("?"):
                    step_lines[-1] += " " + line.strip(); end += len(line)          # wrapped step
                else:
                    break
        danger = re.search(r"^\s*DANGER SIGNS\s*:\s*(.+?)\s*$", after, re.M | re.I)
        prev_end = a.end() + max(end, danger.end() if danger else 0)
        action = norm_action(a.group(1))
        key = question.lower()
        if not (action and step_lines) or len(question) < 15 or key in seen or question.upper().startswith("WHAT TO DO"):
            continue
        seen.add(key)
        answer = (f"ACTION: {action}\nWHAT TO DO:\n" + "\n".join(step_lines) +
                  f"\nDANGER SIGNS: {danger.group(1).strip() if danger else 'none listed'}" +
                  f"\nSOURCE: {chunk['source']}")
        items.append({"chunk_id": chunk["chunk_id"], "section_id": chunk["section_id"],
                      "source": chunk["source"], "kind": kind, "lang": "en",
                      "question": question, "action": action, "answer": answer})
    return items

# self-check on a hand-written sample (no GPU needed)
_sample = """Q: A 3-year-old girl has fever and is breathing fast for 2 days. What should I do?
Action: Refer
WHAT TO DO:
1. Give first dose of amoxicillin.
2. Refer to the health facility.
DANGER SIGNS: chest indrawing, unable to drink
###
Q: A 10-month-old boy has convulsions with fever. What should I do?
ACTION: URGENT REFERRAL
WHAT TO DO:
1. Keep the airway clear.
2. Refer immediately.
###
Q: bad
ACTION: TREAT
WHAT TO DO:
1. x"""
_c = {"chunk_id": "T-1", "section_id": "2.3", "source": "CHEW Standing Orders 2024, Section 2.3 Fever, p. 50"}
_p = parse(_sample, _c, "vignette")
assert [x["action"] for x in _p] == ["REFER", "URGENT"], _p
assert _p[1]["answer"].endswith("SOURCE: CHEW Standing Orders 2024, Section 2.3 Fever, p. 50")
assert "DANGER SIGNS: none listed" in _p[1]["answer"]
_md = """### Case 1
**Case Story:** A 5-month-old girl has a swollen head after a fall. What should I do?

**Action:** REFERRAL
**What to Do:**
1. Refer her today.
**Danger Signs:** convulsions, refusal to feed
"""
_m = parse(_md, _c, "vignette")
assert len(_m) == 1 and _m[0]["action"] == "REFER" and _m[0]["question"].startswith("A 5-month-old"), _m
print("parser self-check passed")

## 4. Load N-ATLaS (float16) and the batched `chat()` helper
Loaded exactly like `01b` and the official model-card example: plain `transformers`, **float16**,
`device_map="auto"` — on Kaggle's GPU T4 x2 the weights are split across both cards; on a single
15 GB card the overflow layers spill onto the CPU (works, but generation is slower).
Generation itself is unchanged: greedy decoding, `repetition_penalty=1.1`.
`chat()` is reused by later notebooks (translation, evaluation).

In [ ]:
import gc, sys, torch, transformers
from packaging import version
# Safe to re-run: free any model already loaded in this session (and the reference an earlier
# failed load keeps in its traceback) before loading again, otherwise the T4 runs out of memory.
for _name in ("trainer", "model"):
    globals().pop(_name, None)
sys.last_type = sys.last_value = sys.last_traceback = None
gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.is_available(), "No GPU: Kaggle -> Accelerator GPU T4 x2; Colab -> Runtime -> Change runtime type"
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.mem_get_info(0)[0] / 2**30:.1f} GB free; overflow layers spill to CPU)")

# the same load as 01b and the N-ATLaS model card: plain float16, no quantisation.
# transformers >= 4.56 calls it `dtype`; older versions use `torch_dtype`.
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
from transformers import AutoModelForCausalLM, AutoTokenizer
load_kw = {DTYPE_KW: torch.float16, "device_map": "auto", "token": HF_TOKEN, "low_cpu_mem_usage": True}
print("loading", BASE_MODEL, "in float16 ...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kw)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model.eval()
model.generation_config.max_length = None           # avoids the "max_new_tokens and max_length" warning
tok.padding_side = "left"
if tok.pad_token is None or tok.pad_token == tok.eos_token:
    tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() else tok.eos_token
INPUT_DEVICE = model.get_input_embeddings().weight.device

def chat(user_msgs, system=None, max_new=MAX_NEW):
    texts = []
    for u in user_msgs:
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": u}]
        texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(INPUT_DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

print(chat(["In one sentence: what is a danger sign of pneumonia in a child?"], max_new=60)[0])

## 5. Build the job list and resume state
Jobs are sorted by chunk length so each batch has similar lengths (less padding, faster).
If `drafts_raw_{PASS}.jsonl` already exists locally or on Hugging Face, finished chunks are skipped.

In [ ]:
n_probe = sum(c["chunk_id"] in probe for c in chunks)
n_per_probe = min(30, max(N_VIGNETTE, -(-PROBE_TARGET // max(n_probe, 1))))   # ceil, capped at 30
if PASS == "vignette":
    jobs = [(c, "vignette", n_per_probe if c["chunk_id"] in probe else N_VIGNETTE) for c in chunks]
elif PASS == "qa":
    jobs = [(c, "qa", N_QA) for c in chunks if c["chunk_id"] not in probe]   # never draft probe Q&A
else:
    raise ValueError("PASS must be 'vignette' or 'qa'")
jobs.sort(key=lambda j: len(j[0]["text"]))
if LIMIT:
    jobs = jobs[:LIMIT]
    print(f"*** SMOKE TEST: only {LIMIT} chunks. Set LIMIT = None in Config for the full run. ***")

RAW_FILE = f"drafts_raw_{PASS}.jsonl"
done = {}
if not os.path.exists(RAW_FILE) and UPLOAD:
    try:
        from huggingface_hub import hf_hub_download
        import shutil
        shutil.copy(hf_hub_download(DATA_REPO, f"drafts/{RAW_FILE}", repo_type="dataset", token=HF_TOKEN), RAW_FILE)
        print("Resuming from Hugging Face copy of", RAW_FILE)
    except Exception:
        pass
if os.path.exists(RAW_FILE):
    for l in open(RAW_FILE, encoding="utf-8"):
        r = json.loads(l)
        done[r["chunk_id"]] = r
todo = [j for j in jobs if j[0]["chunk_id"] not in done]
print(f"{len(jobs)} jobs | {len(done)} already done | {len(todo)} to run")

## 6. Generate (the long step)

In [ ]:
def _retry_upload(api, progress, **kw):
    # Hugging Face can answer 429/500 when several jobs commit at once: retry with backoff;
    # a failed PROGRESS save only warns (the run continues and saves again later)
    import time
    for i in range(6):
        try:
            return api.upload_file(**kw)
        except Exception as e:
            if i == 5:
                if progress:
                    print(f"WARNING: progress upload failed ({type(e).__name__}); continuing"); return None
                raise
            print(f"upload failed ({type(e).__name__}): retry {i + 1}/5 in {15 * 2 ** i}s", flush=True)
            time.sleep(15 * 2 ** i)

def upload(files, msg, progress=False):
    if not UPLOAD:
        return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in files:
        _retry_upload(api, progress, path_or_fileobj=f, path_in_repo=f"drafts/{f}", repo_id=DATA_REPO,
                      repo_type="dataset", commit_message=msg)

prompts_for = {"vignette": VIGNETTE_PROMPT, "qa": QA_PROMPT}
t0, n_batches = time.time(), (len(todo) + BATCH - 1) // BATCH
with open(RAW_FILE, "a", encoding="utf-8") as f:
    for b in range(n_batches):
        batch = todo[b * BATCH:(b + 1) * BATCH]
        prompts = [prompts_for[k].format(source=c["source"], text=c["text"], n=n) for c, k, n in batch]
        try:
            raws = chat(prompts)
        except torch.cuda.OutOfMemoryError:           # long prompt did not fit: retry one by one
            torch.cuda.empty_cache()
            print(f"  OOM in batch {b + 1}: retrying {len(batch)} items individually", flush=True)
            raws = [chat([p])[0] for p in prompts]
        for (c, k, n), raw in zip(batch, raws):
            row = {"chunk_id": c["chunk_id"], "kind": k, "n_requested": n, "raw": raw}
            done[c["chunk_id"]] = row
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        el = time.time() - t0
        print(f"batch {b + 1}/{n_batches}  elapsed {el / 60:.1f} min  "
              f"ETA {el / (b + 1) * (n_batches - b - 1) / 60:.0f} min", flush=True)
        if (b + 1) % SAVE_EVERY == 0:
            upload([RAW_FILE], f"02_draft {PASS}: progress {b + 1}/{n_batches}", progress=True)
upload([RAW_FILE], f"02_draft {PASS}: generation finished")
print("generation done")

## 7. Parse, report, and look at the drafts
Read the printed examples before moving on. If most rows are dropped, the model is not following the layout: lower `N_*` or `BATCH`, or shorten `MAX_WORDS` in 01_extract.

In [ ]:
import collections
by_id = {c["chunk_id"]: c for c in chunks}
drafts, empty = [], []
for cid, r in done.items():
    if cid not in by_id or r["kind"] != PASS:
        continue
    items = parse(r["raw"], by_id[cid], r["kind"])
    drafts += items
    if not items:
        empty.append(cid)

print(f"{len(drafts)} parsed items from {len(done)} chunks; {len(empty)} chunks gave nothing")
print("per chunk (requested vs parsed):",
      f"{sum(r['n_requested'] for r in done.values() if r['kind'] == PASS)} vs {len(drafts)}")
print("ACTION mix:", dict(collections.Counter(d["action"] for d in drafts)))
for d in random.Random(0).sample(drafts, min(3, len(drafts))):
    print("\n---", d["chunk_id"], "---\nQ:", d["question"], "\n" + d["answer"])
if empty:
    print("\n=== raw output of a chunk that parsed to nothing:", empty[0], "===\n", done[empty[0]]["raw"][:1500])

## 8. Export the review sheet
Split rules (runbook Part 3.5):
* run 1 (`vignette`): vignettes from **probe** chunks → `probe`; half of the rest → `bench`; the other half → `train`.
* run 2 (`qa`): everything → `train`.

Import `review_en_{PASS}.csv` into Google Sheets (File → Import → Upload). Make two tabs, `bench_en`
(split = bench or probe) and `train_en`, and follow the review rules in runbook Part 4.

In [ ]:
import csv
rng = random.Random(7)
for d in drafts:
    d["split"] = "train"
if PASS == "vignette":
    for d in drafts:
        if d["chunk_id"] in probe:
            d["split"] = "probe"
    rest = [d for d in drafts if d["split"] == "train"]
    for d in rng.sample(rest, len(rest) // 2):
        d["split"] = "bench"

prefix = "EN-V" if PASS == "vignette" else "EN-Q"
drafts.sort(key=lambda d: (d["split"], d["chunk_id"]))
COLS = ["id", "split", "kind", "chunk_id", "source", "question", "action", "answer",
        "key_facts", "status", "reviewer", "comment"]
OUT = f"review_en_{PASS}.csv"
with open(OUT, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=COLS)
    w.writeheader()
    for i, d in enumerate(drafts):
        w.writerow({**{k: d.get(k, "") for k in COLS}, "id": f"{prefix}{i:05d}"})

open("system_prompt.txt", "w", encoding="utf-8").write(SYSTEM)
print(dict(collections.Counter(d["split"] for d in drafts)), "->", OUT)
if PASS == "vignette":
    n_bench = sum(d["split"] in ("bench", "probe") for d in drafts)
    print(f"benchmark candidates: {n_bench} (plan target: 700+ drafts -> 600 reviewed + 60 probe)")

upload([OUT], f"02_draft {PASS}: review sheet")
if UPLOAD:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="system_prompt.txt", path_in_repo="config/system_prompt.txt",
                                      repo_id=DATA_REPO, repo_type="dataset",
                                      commit_message="answer-format contract")

## Done when
* Run 1: `drafts/review_en_vignette.csv` on Hugging Face with **700+** benchmark/probe candidates (plan: *Sat 3 Oct, 700+ draft benchmark items*). Reviewers start on `bench_en` right away.
* Run 2: `drafts/review_en_qa.csv` with about 1,900 training drafts, ready Sunday.

**Next:** Part 4 (human review in Google Sheets), then Part 5 (`03_translate`) on Sun 4 Oct.